# GO Stanford → Cognitive Map Notebook
This notebook trains a visual backbone, a simple world model, and a tiny TEM-like memory placeholder on image trajectories to produce embeddings and visualize a cognitive map.


In [8]:
import os, math, random
from glob import glob
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device=', device)


device= cuda


## 1. Dataset (trajectory loader)
Expect directory: ROOT/train/positive_L/*.jpg (or positive_R)


In [9]:
class GoStanfordTrajectoryDataset(Dataset):
    def __init__(self, root, split='', side='L', seq_len=5, transform=None, max_items=None):
        self.root = root; self.seq_len = seq_len; self.transform = transform
        self.img_dir = os.path.join(root, split, f'positive_{side}')
        print('Loading images from', self.img_dir)
        self.frames = sorted(glob(os.path.join(self.img_dir, '*.jpg')))
        if max_items is not None:
            self.frames = self.frames[:max_items]
        assert len(self.frames) > seq_len, 'Not enough images.'
    def __len__(self): return len(self.frames) - self.seq_len
    def __getitem__(self, idx):
        imgs = []
        for i in range(idx, idx + self.seq_len):
            img = Image.open(self.frames[i]).convert('RGB')
            if self.transform: img = self.transform(img)
            imgs.append(img)
        return torch.stack(imgs)


## 2. Backbone (embedding only)


In [10]:
class CognitiveBackbone(nn.Module):
    def __init__(self, arch='resnet18', pretrained=True, embed_dim=256):
        super().__init__()
        if arch=='resnet18': net=models.resnet18(pretrained=pretrained); dim=512
        elif arch=='resnet50': net=models.resnet50(pretrained=pretrained); dim=2048
        else: raise ValueError
        self.encoder=nn.Sequential(*list(net.children())[:-2])
        self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.projector=nn.Sequential(nn.Linear(dim,512), nn.ReLU(), nn.Linear(512, embed_dim))
    def forward(self,x):
        f=self.encoder(x); f=self.pool(f).flatten(1)
        z=F.normalize(self.projector(f), dim=1)
        return z


## 3. World Model (z_t → z_{t+1})


In [11]:
class WorldModel(nn.Module):
    def __init__(self, d=256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d, 512),
            nn.ReLU(),
            nn.Linear(512, d)
        )

    def forward(self, z):
        return self.net(z)

## 4. Tiny TEM-like memory (placeholder)
This is a simple differentiable memory that accumulates embeddings as a map.


In [12]:
class TinyTEM(nn.Module):
    def __init__(self, d=256, capacity=1024): super().__init__(); self.capacity=capacity; self.reset()
    def reset(self): self.mem=[]
    def forward(self, z_t):
        # append
        self.mem.append(z_t.detach().cpu())
        if len(self.mem) > self.capacity: self.mem = self.mem[-self.capacity:]
        return z_t
    def get_map(self):
        if len(self.mem)==0: return None
        return torch.cat(self.mem, dim=0)


## 5. Training


In [13]:
ROOT = os.environ.get('GOSTANFORD_ROOT', '/media/zhen/Data/Datasets/nomad_data/go_stanford')
print('Using dataset root:', ROOT)
SEQ=5; BATCH=6; EPOCH=5
tfm=T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize([.485,.456,.406],[.229,.224,.225])])
ds=GoStanfordTrajectoryDataset(ROOT, seq_len=SEQ, transform=tfm, max_items=300)
dl=DataLoader(ds, batch_size=BATCH, shuffle=True)

enc=CognitiveBackbone(embed_dim=256).to(device)
wm=WorldModel(256).to(device)
tem=TinyTEM(256)
opt=torch.optim.Adam(list(enc.parameters())+list(wm.parameters()), lr=1e-4)
mse=nn.MSELoss()

for ep in range(EPOCH):
    tot=0
    for seq in dl:
        B,T,C,H,W = seq.shape
        seq=seq.to(device)
        z=enc(seq.view(B*T,C,H,W)).view(B,T,-1)
        pred=wm(z[:,:-1])
        loss=mse(pred, z[:,1:])
        opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item()
    print('epoch',ep,'loss',tot/len(dl))


Using dataset root: /media/zhen/Data/Datasets/nomad_data/go_stanford
Loading images from /media/zhen/Data/Datasets/nomad_data/go_stanford/positive_L


AssertionError: Not enough images.

## 6. Build Cognitive Map and Visualize


In [ ]:
# Run a single long trajectory to build map
tem.reset()
sample = ds[0].unsqueeze(0).to(device)
B,T,C,H,W = sample.shape
with torch.no_grad():
    z = enc(sample.view(B*T,C,H,W)).view(B,T,-1)[0]
    for t in range(T): tem(z[t:t+1])
MAP = tem.get_map().numpy()
print('map size', MAP.shape)

Z2 = PCA(2).fit_transform(MAP)
plt.figure(figsize=(5,4)); plt.plot(Z2[:,0], Z2[:,1], '-o'); plt.title('Cognitive Map (PCA)'); plt.axis('equal'); plt.show()
